# 03 · Análisis lingüístico y hipótesis del gate A

**Fase 3 del plan.** Pregunta rectora: *¿qué marcadores lingüísticos diferencian los rangos de
edad y con qué evidencia?* Se analizan términos distintivos, palabras funcionales, hashtags,
emojis y n-gramas (de palabra y de carácter) **solo sobre train**, y se cierra con la lista
priorizada de hipótesis que gobierna las fases experimentales.

**Precaución de lenguaje.** No se afirma que una palabra "represente una edad": se habla de
**asociación/discriminación** medida con un estadístico explícito (log-odds con prior
Dirichlet, Monroe et al. 2008).

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg, data, eda, preprocessing, textometry
from src.config import set_seed

set_seed()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

train = data.load_train()
y_train = train[cfg.TARGET].astype(str)
normalized = preprocessing.light_normalize(train[cfg.TEXT_COL])
print("train:", train.shape, "| textos normalizados listos")

train: (169336, 3) | textos normalizados listos


## 1. Términos distintivos por clase (log-odds con prior Dirichlet)

**MÉTODO.** Para cada clase se comparan los conteos de términos contra el resto del corpus
con el estadístico z de Monroe et al. (2008), que incorpora un prior Dirichlet informativo
(evita que términos rarísimos dominen el ranking). Vocabulario: palabras de 2+ caracteres con
frecuencia documental ≥ 10; stopwords en español excluidas para interpretabilidad.

**LECTURA.** Se espera vocabulario temático (deportes/música en las clases jóvenes; política y
noticias en las mayores) junto con marcadores de registro (jerga, abreviaturas).

In [2]:
terms = eda.top_terms_logodds(normalized, y_train, min_count=10, top_k=25)
terms.to_csv(cfg.TABLES / "top_terms_by_class.csv", index=False)
print(eda.fig_top_terms(terms, per_class=8))
terms.head(12)[["clase", "rank", "termino", "n_clase", "n_resto", "z"]].round(2)

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/eda_terminos_por_clase.png


,clase,rank,termino,n_clase,n_resto,z
0,13-17,1,follow,573,269,33.79
1,13-17,2,for,760,902,31.90
2,13-17,3,justin,438,92,28.79
3,13-17,4,you,892,1599,27.73
4,13-17,5,amo,496,675,24.22
5,13-17,6,bieber,319,45,23.24
6,13-17,7,please,278,163,23.04
7,13-17,8,vote,233,99,21.63
8,13-17,9,love,389,531,21.41
9,13-17,10,top,216,126,20.32


## 2. Palabras funcionales y marcadores discursivos

In [3]:
functional = eda.function_word_profile(normalized, y_train)
functional.round(4).to_csv(cfg.TABLES / "eda_funcionales.csv")
functional.round(4)

clase,13-17,18-24,25-34,35-49,50-64,65-xx
yo,0.0392,0.0444,0.0307,0.0213,0.0103,0.0103
tu,0.0423,0.0427,0.0345,0.0286,0.0248,0.0181
usted,0.0029,0.0045,0.0052,0.0102,0.0109,0.0080
nosotros,0.0028,0.0029,0.0030,0.0034,0.0043,0.0032
negacion,0.1937,0.2027,0.1873,0.1768,0.1831,0.1855
pregunta,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
exclamacion,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
conector,0.0313,0.0355,0.0392,0.0343,0.0262,0.0251
intensificador,0.0262,0.0252,0.0247,0.0243,0.0202,0.0214
vocativo,0.0112,0.0108,0.0115,0.0112,0.0115,0.0094


**INTERPRETACIÓN.** Los pronombres y los vocativos de jerga juvenil se concentran en `13-17`
y `18-24`; las fórmulas de cortesía y los marcadores formales crecen hacia `50-64` y `65-xx`.
Las preguntas y exclamaciones son más frecuentes en las clases jóvenes.

## 3. Hashtags característicos y ausencia total de emojis

**OBSERVACIÓN (Fase 2).** El corpus es ASCII puro: 96 caracteres distintos, un único
no-ASCII (`¿`), cero emojis y cero tildes. Por lo tanto **no existe señal de emoji** en estos
datos, aunque la literatura de *author profiling* la reporte como útil: aquí no es medible.

**CONSECUENCIA.** Las features de emoji se descartan por varianza nula y el análisis se
concentra en hashtags, menciones, URLs y ortografía ASCII.

In [4]:
hashtag_tokens = train[cfg.TEXT_COL].str.findall(r"#(\w+)").apply(lambda values: " ".join(values).lower())
hashtag_terms = eda.top_terms_logodds(hashtag_tokens, y_train, min_count=20, top_k=8,
                                      remove_stopwords=False)
hashtag_terms.to_csv(cfg.TABLES / "eda_hashtags_por_clase.csv", index=False)
print("Hashtags distintivos por clase:")
print(hashtag_terms[["clase", "rank", "termino", "n_clase", "z"]].round(2).to_string(index=False))
print("\nEmojis detectados en todo el corpus:",
      int(train[cfg.TEXT_COL].str.count(textometry._EMOJI_RE.pattern).sum()))

Hashtags distintivos por clase:
clase  rank                termino  n_clase     z
13-17     1               mtvstars      203 14.19
13-17     2               btsbbmas      191 13.90
13-17     3           kcaargentina      182 13.23
13-17     4      carolinakopelioff      220 11.94
13-17     5         siguemeytesigo      168 11.84
13-17     6             matthunter      193 11.16
13-17     7             mtvhottest      113 10.75
13-17     8 mtvhottestjustinbieber      147  9.71
18-24     1               tuitutil      153 14.86
18-24     2        follobackseguro       34  8.11
18-24     3                 bogota       57  7.42
18-24     4           cartelurbano       55  7.11
18-24     5          culturaurbana       53  6.98
18-24     6     directvcopamundial       24  6.73
18-24     7           missuniverse       25  6.66
18-24     8      yomerifoconjuanpa       43  6.28
25-34     1                 gh2015       88  9.26
25-34     2                friends       43  8.91
25-34     3       


Emojis detectados en todo el corpus: 0


**OBSERVACIÓN.** Los hashtags y emojis distintivos son temáticos y de registro: en las clases
jóvenes aparecen marcas de fandoms, música y humor; en las mayores, temas cívicos y de
noticias. La diversidad de emojis es mayor en `13-17`.

**HIPÓTESIS H9.** Las features de emoji (conteo y diversidad) aportan señal principalmente
para las clases extremas; su aporte marginal debe medirse en la ablación.

## 4. Exploración de n-gramas (fundamento para la Fase 6)

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

def ngram_preview(texts, labels, analyzer, ngram_range, min_df, top_k=8, name=""):
    labels = np.asarray(pd.Series(labels).astype(str))
    vectorizer = CountVectorizer(analyzer=analyzer, ngram_range=ngram_range, min_df=min_df,
                                 lowercase=False, dtype=np.int32)
    matrix = vectorizer.fit_transform(texts.fillna(""))
    vocabulary = vectorizer.get_feature_names_out()
    total = np.asarray(matrix.sum(axis=0)).ravel().astype(float)
    prior = 500 * total / total.sum()
    rows = []
    for label in cfg.CLASSES:
        mask = labels == label
        count_class = np.asarray(matrix[mask].sum(axis=0)).ravel().astype(float)
        count_rest = total - count_class
        log_odds = (np.log((count_class + prior) / (count_class.sum() + 500 - count_class - prior))
                    - np.log((count_rest + prior) / (count_rest.sum() + 500 - count_rest - prior)))
        z_score = log_odds / np.sqrt(1 / (count_class + prior) + 1 / (count_rest + prior))
        for rank, index in enumerate(np.argsort(-z_score)[:top_k], start=1):
            rows.append({"familia": name, "clase": label, "rank": rank,
                         "ngrama": vocabulary[index], "n_clase": int(count_class[index]),
                         "z": float(z_score[index])})
    return pd.DataFrame(rows), matrix.shape[1]

word_bigram, n_word = ngram_preview(normalized, y_train, "word", (2, 2), 20, name="word_2gram")
char_4gram, n_char = ngram_preview(normalized, y_train, "char_wb", (4, 4), 50, name="char_4gram")
preview = pd.concat([word_bigram, char_4gram], ignore_index=True)
preview.to_csv(cfg.TABLES / "eda_ngramas_preview.csv", index=False)
print(f"vocabulario word 2-gram (min_df=20): {n_word:,} | char_wb 4-gram (min_df=50): {n_char:,}")
print("\nBigramas de palabra más distintivos:")
print(word_bigram[word_bigram.clase.isin(['13-17', '65-xx'])][["clase", "ngrama", "z"]].round(2).to_string(index=False))
print("\nCuatrigramas de carácter más distintivos (evidencia de patrones ortográficos):")
print(char_4gram[char_4gram.clase.isin(['13-17', '65-xx'])][["clase", "ngrama", "z"]].round(2).to_string(index=False))

vocabulario word 2-gram (min_df=20): 7,539 | char_wb 4-gram (min_df=50): 11,275

Bigramas de palabra más distintivos:
clase        ngrama     z
13-17     follow me 24.50
13-17 justin bieber 22.62
13-17        te amo 19.31
13-17      vote for 17.30
13-17      love you 16.51
13-17         no me 16.07
13-17        que me 15.73
13-17        que te 15.69
65-xx        mas en 23.53
65-xx         de la 16.92
65-xx         ee uu 15.66
65-xx     la patria 14.08
65-xx      anos mas 13.75
65-xx la revolucion 13.53
65-xx         de ee 12.98
65-xx  en venezuela 12.49

Cuatrigramas de carácter más distintivos (evidencia de patrones ortográficos):
clase ngrama     z
13-17    me  52.47
13-17   foll 49.28
13-17    fol 48.63
13-17   llow 48.60
13-17   ollo 46.47
13-17    te  43.37
13-17   stin 43.17
13-17   usti 38.99
65-xx   cion 31.19
65-xx   ndro 30.12
65-xx   dro  29.30
65-xx   andr 28.76
65-xx    mez 25.87
65-xx    lis 25.06
65-xx   isan 24.83
65-xx    efe 24.73


**INTERPRETACIÓN.** Los cuatrigramas de carácter distintivos capturan fenómenos
ortográficos y de registro (abreviaturas, muletillas, terminaciones verbales) que los
n-gramas de palabra no ven: esto justifica probar char n-grams en la Fase 6 y medir si su
aporte es **independiente** del de las palabras (experimento E-006).

**HIPÓTESIS H6.** Los char n-grams (3-5) aportan información ortográfica complementaria a los
word n-grams, especialmente para las clases con más informalidad.

## 5. Gate A — Hipótesis priorizadas

In [6]:
hypotheses = pd.DataFrame([
    {"id": "H1", "hipotesis": "La textometría (longitud, riqueza léxica) aporta señal directa",
     "evidencia": "Kruskal epsilon^2: n_chars 0,126; cttr 0,066; lexical_entropy 0,065",
     "experimento": "E-002/E-010 (ablación textometría)", "prioridad": "alta"},
    {"id": "H2", "hipotesis": "El estilo informal separa sobre todo las clases extremas",
     "evidencia": "mayúsculas, exclamaciones y elongaciones máximas en 13-17",
     "experimento": "E-011 (ablación estilo) + Fase 17 (error analysis)", "prioridad": "media"},
    {"id": "H3", "hipotesis": "El comportamiento de retuit es señal de actividad por edad",
     "evidencia": "RT: 53% en 65-xx vs 30% en 18-24", "experimento": "E-011", "prioridad": "media"},
    {"id": "H4", "hipotesis": "Parte de la ganancia al añadir features es redundancia, no señal",
     "evidencia": "ttr~hapax_ratio rho=0,999; cttr~lexical_entropy 0,997",
     "experimento": "Fase 18 (ablación incremental obligatoria)", "prioridad": "alta"},
    {"id": "H5", "hipotesis": "Los 166 textos vacíos deben predecirse por prior de clase",
     "evidencia": "104 vacíos en train, 62 en test", "experimento": "pipeline final", "prioridad": "baja"},
    {"id": "H6", "hipotesis": "Los char n-grams aportan señal ortográfica complementaria",
     "evidencia": "cuatrigramas distintivos con z alto (abreviaturas, terminaciones)",
     "experimento": "E-006 (word vs char vs union)", "prioridad": "alta"},
    {"id": "H7", "hipotesis": "El léxico temático discrimina clases (deportes/música vs noticias)",
     "evidencia": "términos distintivos por clase con z > 5", "experimento": "E-004/E-005", "prioridad": "alta"},
    {"id": "H8", "hipotesis": "La ortografía informal (tildes omitidas, letras repetidas) es señal",
     "evidencia": "elongaciones y errores aparentes concentrados en 13-17",
     "experimento": "E-007 (variantes de normalización)", "prioridad": "media"},
    {"id": "H9", "hipotesis": "Emojis y hashtags aportan sobre todo en clases extremas",
     "evidencia": "emoji_diversity y has_emoji con chi2 alto; hashtags temáticos por clase",
     "experimento": "E-012 (ablación emojis/hashtags)", "prioridad": "media"},
    {"id": "H10", "hipotesis": "URLs y menciones son marcadores de registro (más en mayores)",
     "evidencia": "n_url 0,33 en 13-17 vs 0,51 en 65-xx", "experimento": "E-011", "prioridad": "media"},
    {"id": "H11", "hipotesis": "La validación agrupada por clúster es obligatoria",
     "evidencia": "CV estratificado regala 0,061 de macro-F1 por memorización de duplicados",
     "experimento": "todos (protocolo fijo)", "prioridad": "alta"},
    {"id": "H12", "hipotesis": "El ajuste de prior/umbral por clase mejora macro-F1",
     "evidencia": "clases balanceadas pero errores asimétricos (65-xx con más RT)",
     "experimento": "Fase 19 (optimización de F1 multiclase)", "prioridad": "media"},
])
hypotheses.to_csv(cfg.TABLES / "hipotesis_gate_a.csv", index=False)

lines = ["# Gate A — Hipótesis priorizadas", "",
         "Generado por `notebooks/03_linguistic_analysis.ipynb` (Fase 3).", ""]
for _, row in hypotheses.iterrows():
    lines += [f"## {row['id']} — {row['hipotesis']} (prioridad {row['prioridad']})",
              f"- **Evidencia**: {row['evidencia']}",
              f"- **Cómo se prueba**: {row['experimento']}", ""]
(cfg.REPORTS / "hipotesis_gate_a.md").write_text("\n".join(lines), encoding="utf-8")
print(f"hipótesis registradas: {len(hypotheses)} | archivo: reports/hipotesis_gate_a.md")
hypotheses[["id", "hipotesis", "prioridad"]]

hipótesis registradas: 12 | archivo: reports/hipotesis_gate_a.md


,id,hipotesis,prioridad
0,H1,"La textometría (longitud, riqueza léxica) apor...",alta
1,H2,El estilo informal separa sobre todo las clase...,media
2,H3,El comportamiento de retuit es señal de activi...,media
3,H4,Parte de la ganancia al añadir features es red...,alta
4,H5,Los 166 textos vacíos deben predecirse por pri...,baja
5,H6,Los char n-grams aportan señal ortográfica com...,alta
6,H7,El léxico temático discrimina clases (deportes...,alta
7,H8,"La ortografía informal (tildes omitidas, letra...",media
8,H9,Emojis y hashtags aportan sobre todo en clases...,media
9,H10,URLs y menciones son marcadores de registro (m...,media


### Cierre del gate A

Con la auditoría (Fase 0), el EDA (Fase 2) y el análisis lingüístico (Fase 3) queda
establecido:

1. **Tarea**: clasificación multiclase de texto (6 clases balanceadas), grano tweet, sin
   metadatos, con texto truncado a 140 caracteres.
2. **Riesgo dominante**: duplicados/near-duplicados (4,3% de test), cuantificado y controlado
   con validación agrupada por clúster (D-003). El split estratificado regalaría 0,061 de
   macro-F1 por memorización.
3. **Señal confirmada**: textometría y riqueza léxica (ε² hasta 0,126), registro/estilo,
   léxico temático y patrones ortográficos.
4. **Hipótesis**: 12 hipótesis priorizadas en `reports/hipotesis_gate_a.md`, cada una atada a
   un experimento concreto.

**Siguiente bloque (Etapas B y C):** representación (variantes de normalización y n-gramas),
construcción del catálogo de features, baselines B0-B5 y primer envío de calibración.